In [16]:
ensure project root is importable
import sys
from pathlib import Path
project_root = Path.cwd().parent  # if notebook is inside gast_project/notebooks
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# import the class
from scripts.sra_extractor import SRAExtractor

SyntaxError: invalid syntax (3720653925.py, line 1)

In [8]:
import os
import pandas as pd
import subprocess
from tqdm import tqdm

class SRAExtractor:
    def __init__(self, project_dir="./gast_test", ena_preferred=False):
        self.project_dir = os.path.abspath(project_dir)
        self.ena_preferred = ena_preferred
        self.metadata_dir = os.path.join(self.project_dir, "data", "metadata")
        self.sequence_dir = os.path.join(self.project_dir, "data", "sequences")
        os.makedirs(self.metadata_dir, exist_ok=True)
        os.makedirs(self.sequence_dir, exist_ok=True)
        print(f"[INFO] Project initialized at {self.project_dir}")

    def fetch_metadata(self, organism="Klebsiella pneumoniae", max_results=10):
        """Fetch metadata from NCBI SRA."""
        from Bio import Entrez
        Entrez.email = "your_email@example.com"

        print(f"[INFO] Fetching metadata for {organism}...")
        handle = Entrez.esearch(db="sra", term=organism, retmax=max_results)
        record = Entrez.read(handle)
        ids = record["IdList"]
        handle.close()

        if not ids:
            print("[WARNING] No records found.")
            return None

        summaries = []
        for sra_id in tqdm(ids, desc="Fetching summaries"):
            handle = Entrez.esummary(db="sra", id=sra_id)
            summary = Entrez.read(handle)
            summaries.append(summary)
            handle.close()

        data = []
        for summary in summaries:
            for docsum in summary:
                runinfo = {
                    "SRA_ID": docsum.get("Id"),
                    "Title": docsum.get("Title"),
                    "Runs": docsum.get("Runs"),
                    "Exp_XML": docsum.get("ExpXml"),
                }
                data.append(runinfo)

        df = pd.DataFrame(data)
        csv_path = os.path.join(self.metadata_dir, f"SraRunInfo_{organism.replace(' ', '_')}.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Metadata saved to {csv_path}")
        return df

    def download_runs(self, run_ids):
        """Download selected runs using prefetch + fasterq-dump (from SRA Toolkit)."""
        print("[INFO] Starting downloads...")

        for run_id in run_ids:
            out_dir = os.path.join(self.sequence_dir, run_id)
            os.makedirs(out_dir, exist_ok=True)

            fastq_file_1 = os.path.join(out_dir, f"{run_id}_1.fastq.gz")
            fastq_file_2 = os.path.join(out_dir, f"{run_id}_2.fastq.gz")

            if os.path.exists(fastq_file_1) and os.path.exists(fastq_file_2):
                print(f"[SKIP] {run_id} already exists, skipping.")
                continue

            print(f"[INFO] Downloading {run_id}...")
            try:
                subprocess.run(["prefetch", run_id], check=True)
                subprocess.run(["fasterq-dump", "--split-files", run_id, "-O", out_dir], check=True)
                subprocess.run(["gzip", os.path.join(out_dir, f"{run_id}_1.fastq")], check=True)
                subprocess.run(["gzip", os.path.join(out_dir, f"{run_id}_2.fastq")], check=True)
                print(f"[DONE] {run_id} downloaded successfully.")
            except subprocess.CalledProcessError as e:
                print(f"[ERROR] Download failed for {run_id}: {e}")

        print("[INFO] All downloads completed.")


In [9]:
# set project root and your email (NCBI requires a valid email)
#ex = SRAExtractor(project_root=".", email="dadaoyadokun@gmail.com", default_retmax=100)
print("Project metadata dir:", ex.data_meta)
print("Project sequences dir:", ex.data_seq)

Project metadata dir: /home/olasu/project_gast/notebooks/data/metadata
Project sequences dir: /home/olasu/project_gast/notebooks/data/sequences


In [10]:
# Fetch metadata for Klebsiella pneumoniae (example)
organism = "Klebsiella pneumoniae"
df = ex.fetch_runinfo(organism, retmax=50)   # change retmax as you wish
print("Fetched rows:", len(df))
# keep it in memory
ex.last_metadata = df

Fetched rows: 50


In [11]:
save_path = ex.save_metadata(df, filename="SraRunInfo_kp.csv")
print("Saved metadata to:", save_path)

Saved metadata to: /home/olasu/project_gast/notebooks/data/metadata/SraRunInfo_kp.csv


In [15]:
# show first 10 rows (change n if you want)
ex.preview_metadata(csvpath="SraRunInfo_kp.csv", n=50)

AttributeError: 'SRAExtractor' object has no attribute 'preview_metadata'

In [13]:
ex = SRAExtractor(project_dir="./gast_test", ena_preferred=True)

# Check structure now (this will fix paths)
print(os.listdir(ex.project_dir))

# Download one sequence
ex.download_runs(["SRR35784663"])

[INFO] Project initialized at /home/olasu/project_gast/notebooks/gast_test
['data']
[INFO] Starting downloads...
[SKIP] SRR35784663 already exists, skipping.
[INFO] All downloads completed.


In [2]:
# ensure project root is importable
import sys
from pathlib import Path

project_root = Path.cwd().parent  # if notebook is inside gast_project/notebooks
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
# import the class
from scripts.sra_extractor import SRAExtractor

In [3]:
import os
import pandas as pd
import subprocess
from tqdm import tqdm

class SRAExtractor:
    def __init__(self, project_dir="./gast_test", ena_preferred=False):
        self.project_dir = os.path.abspath(project_dir)
        self.ena_preferred = ena_preferred
        self.metadata_dir = os.path.join(self.project_dir, "data", "metadata")
        self.sequence_dir = os.path.join(self.project_dir, "data", "sequences")
        os.makedirs(self.metadata_dir, exist_ok=True)
        os.makedirs(self.sequence_dir, exist_ok=True)
        print(f"[INFO] Project initialized at {self.project_dir}")

    def fetch_metadata(self, organism="Klebsiella pneumoniae", max_results=10):
        """Fetch metadata from NCBI SRA."""
        from Bio import Entrez
        Entrez.email = "your_email@example.com"

        print(f"[INFO] Fetching metadata for {organism}...")
        handle = Entrez.esearch(db="sra", term=organism, retmax=max_results)
        record = Entrez.read(handle)
        ids = record["IdList"]
        handle.close()

        if not ids:
            print("[WARNING] No records found.")
            return None

        summaries = []
        for sra_id in tqdm(ids, desc="Fetching summaries"):
            handle = Entrez.esummary(db="sra", id=sra_id)
            summary = Entrez.read(handle)
            summaries.append(summary)
            handle.close()

        data = []
        for summary in summaries:
            for docsum in summary:
                runinfo = {
                    "SRA_ID": docsum.get("Id"),
                    "Title": docsum.get("Title"),
                    "Runs": docsum.get("Runs"),
                    "Exp_XML": docsum.get("ExpXml"),
                }
                data.append(runinfo)

        df = pd.DataFrame(data)
        csv_path = os.path.join(self.metadata_dir, f"SraRunInfo_{organism.replace(' ', '_')}.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Metadata saved to {csv_path}")
        return df

    def download_runs(self, run_ids):
        """Download selected runs using prefetch + fasterq-dump (from SRA Toolkit)."""
        print("[INFO] Starting downloads...")

        for run_id in run_ids:
            out_dir = os.path.join(self.sequence_dir, run_id)
            os.makedirs(out_dir, exist_ok=True)

            fastq_file_1 = os.path.join(out_dir, f"{run_id}_1.fastq.gz")
            fastq_file_2 = os.path.join(out_dir, f"{run_id}_2.fastq.gz")

            if os.path.exists(fastq_file_1) and os.path.exists(fastq_file_2):
                print(f"[SKIP] {run_id} already exists, skipping.")
                continue

            print(f"[INFO] Downloading {run_id}...")
            try:
                subprocess.run(["prefetch", run_id], check=True)
                subprocess.run(["fasterq-dump", "--split-files", run_id, "-O", out_dir], check=True)
                subprocess.run(["gzip", os.path.join(out_dir, f"{run_id}_1.fastq")], check=True)
                subprocess.run(["gzip", os.path.join(out_dir, f"{run_id}_2.fastq")], check=True)
                print(f"[DONE] {run_id} downloaded successfully.")
            except subprocess.CalledProcessError as e:
                print(f"[ERROR] Download failed for {run_id}: {e}")

        print("[INFO] All downloads completed.")


In [4]:
# set project root and your email (NCBI requires a valid email)
ex = SRAExtractor(project_root=".", email="dadaoyadokun@gmail.com", default_retmax=100)
print("Project metadata dir:", ex.data_meta)
print("Project sequences dir:", ex.data_seq)

TypeError: SRAExtractor.__init__() got an unexpected keyword argument 'project_root'

In [6]:
# Fetch metadata for Klebsiella pneumoniae (example)
organism = "Klebsiella pneumoniae"
df = ex.fetch_runinfo(organism, retmax=50)   # change retmax as you wish
print("Fetched rows:", len(df))
# keep it in memory
ex.last_metadata = df

NameError: name 'ex' is not defined

In [7]:
save_path = ex.save_metadata(df, filename="SraRunInfo_kp.csv")
print("Saved metadata to:", save_path)


NameError: name 'ex' is not defined

In [8]:
# show first 10 rows (change n if you want)
ex.preview_metadata(csvpath="SraRunInfo_kp.csv", n=50)

NameError: name 'ex' is not defined

In [9]:
ex = SRAExtractor(project_dir="./gast_test", ena_preferred=True)

# Check structure now (this will fix paths)
print(os.listdir(ex.project_dir))

# Download one sequence
ex.download_runs(["SRR35782937"])


[INFO] Project initialized at /home/olasu/project_gast/notebooks/gast_test
['data']
[INFO] Starting downloads...
[SKIP] SRR35782937 already exists, skipping.
[INFO] All downloads completed.


In [ ]:
import os
import pandas as pd
import subprocess
from tqdm import tqdm

class SRAExtractor:
    def __init__(self, project_dir="./gast_test", ena_preferred=False):
        self.project_dir = os.path.abspath(project_dir)
        self.ena_preferred = ena_preferred
        self.metadata_dir = os.path.join(self.project_dir, "data", "metadata")
        self.sequence_dir = os.path.join(self.project_dir, "data", "sequences")
        os.makedirs(self.metadata_dir, exist_ok=True)
        os.makedirs(self.sequence_dir, exist_ok=True)
        print(f"[INFO] Project initialized at {self.project_dir}")

    def fetch_metadata(self, organism="Klebsiella pneumoniae", max_results=10):
        """Fetch metadata from NCBI SRA."""
        from Bio import Entrez
        Entrez.email = "your_email@example.com"

        print(f"[INFO] Fetching metadata for {organism}...")
        handle = Entrez.esearch(db="sra", term=organism, retmax=max_results)
        record = Entrez.read(handle)
        ids = record["IdList"]
        handle.close()

        if not ids:
            print("[WARNING] No records found.")
            return None

        summaries = []
        for sra_id in tqdm(ids, desc="Fetching summaries"):
            handle = Entrez.esummary(db="sra", id=sra_id)
            summary = Entrez.read(handle)
            summaries.append(summary)
            handle.close()

        data = []
        for summary in summaries:
            for docsum in summary:
                runinfo = {
                    "SRA_ID": docsum.get("Id"),
                    "Title": docsum.get("Title"),
                    "Runs": docsum.get("Runs"),
                    "Exp_XML": docsum.get("ExpXml"),
                }
                data.append(runinfo)

        df = pd.DataFrame(data)
        csv_path = os.path.join(self.metadata_dir, f"SraRunInfo_{organism.replace(' ', '_')}.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Metadata saved to {csv_path}")
        return df

    def download_runs(self, run_ids):
        """Download selected runs using prefetch + fasterq-dump (from SRA Toolkit)."""
        print("[INFO] Starting downloads...")

        for run_id in run_ids:
            out_dir = os.path.join(self.sequence_dir, run_id)
            os.makedirs(out_dir, exist_ok=True)

            fastq_file_1 = os.path.join(out_dir, f"{run_id}_1.fastq.gz")
            fastq_file_2 = os.path.join(out_dir, f"{run_id}_2.fastq.gz")

            if os.path.exists(fastq_file_1) and os.path.exists(fastq_file_2):
                print(f"[SKIP] {run_id} already exists, skipping.")
                continue

            print(f"[INFO] Downloading {run_id}...")
            try:
                subprocess.run(["prefetch", run_id], check=True)
                subprocess.run(["fasterq-dump", "--split-files", run_id, "-O", out_dir], check=True)
                subprocess.run(["gzip", os.path.join(out_dir, f"{run_id}_1.fastq")], check=True)
                subprocess.run(["gzip", os.path.join(out_dir, f"{run_id}_2.fastq")], check=True)
                print(f"[DONE] {run_id} downloaded successfully.")
            except subprocess.CalledProcessError as e:
                print(f"[ERROR] Download failed for {run_id}: {e}")

        print("[INFO] All downloads completed.")
